In [ ]:
""" 
Notebook: Simulate the operations performed by different humans throughout the E2E Flow:

    - Customers will place an order from the restaurant screens via an API request.
    - The restaurant staff will either accept or reject the request

EDEM. Master Big Data & Cloud 2025/2026
Professor: Javi Briones
"""

#### AWS Setup

In [ ]:
# Load environment variables from .dev file
from dotenv import load_dotenv

load_dotenv(dotenv_path="../../00_DocAux/.env") 

In [ ]:
# Your AWS Credentials
import os

AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_KEY")
REGION = os.getenv("AWS_REGION", "eu-north-1") 

In [ ]:
import boto3

session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=REGION
)

#### Create an order: Payload

In [ ]:
import random

order_payload = {
  "orderId": random.randint(0,9999),
  "customer": "",
  "items": [
    {
      "id": "1",
      "name": "Taco Supreme"
    },
    {
      "id": "2",
      "name": "Quesadilla"
    }
  ]
}

#### API Request

In [ ]:
API_ENDPOINT = "<YOUR_API_INVOKE_URL>"

In [ ]:
from requests_aws4auth import AWS4Auth
credentials = session.get_credentials().get_frozen_credentials()

auth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    REGION,
    "execute-api",
    session_token=credentials.token,
)

In [ ]:
# Simulate the customer placing an order

import requests
import json

response = requests.post(
    API_ENDPOINT + '/orders',
    headers={
        "Content-Type": "application/json"
    },
    data=json.dumps(order_payload),
    auth=auth
)

print(response.status_code)
print(response.text)

#### Human Approval

In [ ]:
import boto3
import json

In [ ]:
# Initialize Lambda client
lambda_client = session.client('lambda')

In [ ]:
# Replace with the actual token from LambdaHumanApproval logs
task_token = input('task_token')

In [ ]:
# Simulate Approval
payload = {
    "taskToken": task_token,
    "customer": order_payload['customer'],
    "orderId": order_payload['orderId'],
    "approved": True  # Set to False if you want to simulate rejection
}

In [ ]:
# Invoke the approval handler Lambda
response = lambda_client.invoke(
    FunctionName='LambdaApprovalHandler',
    Payload=json.dumps(payload)
)

# Read and print Lambda response
print(response['StatusCode'])
print(response['Payload'].read().decode())